In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [3]:
load_dotenv()

True

In [4]:
model = ChatGroq(model= "llama-3.1-8b-instant")

In [7]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive","negative"]= Field(description="Sentiment of the review")
    


In [8]:
str_model= model.with_structured_output(SentimentSchema)

In [11]:
prompt="what is the sentiment of the following review - The software is too good"
str_model.invoke(prompt).sentiment

'positive'

In [12]:
class ReviewState(TypedDict):
    review : str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [13]:
def find_sentiment(state: ReviewState ):
    prompt= f'for the following review find out the sentiment \n {state['review']}'
    sentiment=str_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

In [17]:
def check_sentiment(state: ReviewState)-> Literal["positive_response", "run_dignosis"]:
    if(state['sentiment']== 'positive'):
        return 'positive_response'
    else:
        return 'run_dignosis'

In [ ]:
graph= StateGraph(ReviewState)
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response',)
graph.add_edge(START,'find_sentiment')
graph.add_edge('find_sentiment',END)
workflow=graph.compile()

In [16]:
intial_state={
    'review': "The product was realy good"
}
workflow.invoke(intial_state)

{'review': 'The product was realy good', 'sentiment': 'positive'}